# Lesson 01 — Linear Regression, Cost Function, Gradient Descent

We build linear regression from nothing but NumPy arrays: no scikit-learn, no autograd.
By the end of this notebook you will have written, and understood the maths behind:

1. the **model** $f_{w,b}(x)$ — the thing that makes predictions,
2. the **cost function** $J(w,b)$ — a single number saying how wrong the model is,
3. the **gradient** $\nabla J$ — which direction makes the cost go *down*,
4. **gradient descent** — walking downhill until we reach the bottom.

The polished versions of every function live in `linear_regression.py`, and
`test_linear_regression.py` checks them. Here we derive them step by step.

---

## Notation

| symbol | meaning |
|---|---|
| $m$ | number of training examples |
| $n$ | number of features |
| $x^{(i)}$ | the feature vector of the $i$-th example, $x^{(i)} \in \mathbb{R}^n$ |
| $y^{(i)}$ | the target (true value) of the $i$-th example |
| $x_j^{(i)}$ | feature $j$ of example $i$ |
| $X$ | the design matrix, shape $(m, n)$ — one **row per example** |
| $w, b$ | the parameters we learn: weights $w \in \mathbb{R}^n$ and bias $b \in \mathbb{R}$ |

Superscript $(i)$ indexes *examples*, subscript $j$ indexes *features*. Keeping those
two apart is most of the battle when reading ML papers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)          # reproducible randomness
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

## 1. A dataset

Let's predict **house price** from **size**. One feature, so $n = 1$ and we can plot
everything. We fabricate the data from a line we choose, then add noise, that way we
know the right answer and can check whether our algorithm recovers it.

The truth we are hiding from the algorithm: $y = 3.5x - 2 + \varepsilon$.

In [ ]:
m = 100
X = rng.uniform(0, 10, size=(m, 1))                       # (m, n) = (100, 1)
y = 3.5 * X[:, 0] - 2.0 + rng.normal(0, 1.5, size=m)      # (m,)

print("X.shape =", X.shape, "   y.shape =", y.shape)
print("first 3 rows of X:", X[:3, 0].round(2), " -> y:", y[:3].round(2))

plt.scatter(X, y, s=18, alpha=0.7)
plt.xlabel("size $x$ (100 sq ft)"); plt.ylabel("price $y$ ($1000s)")
plt.title("Training data"); plt.show()

### Why is $X$ two-dimensional for one feature?

Because we want the *same code* to work for 1 feature or 100. A row is an example, a
column is a feature. NumPy's `@` (matrix multiply) then does the whole dataset at once.
`y` stays 1-D of shape $(m,)$ — a plain vector of targets.

## 2. The model

A linear model predicts with a straight line (a hyperplane when $n > 1$):

$$f_{w,b}(x) = w \cdot x + b = \sum_{j=1}^{n} w_j x_j + b$$

For all $m$ examples at once, this is one matrix–vector product:

$$\hat{y} = Xw + b \qquad
\underbrace{(m,n)}_{X}\;\underbrace{(n,)}_{w} \rightarrow \underbrace{(m,)}_{\hat{y}}$$

The $+b$ is *bias* term — NumPy adds the scalar to every element.

In [ ]:
def predict(X, w, b):
    # model output for every row of X -> shape (m,)
    return X @ w + b


w_guess, b_guess = np.array([2.0]), 0.0     # a deliberately bad guess
y_hat = predict(X, w_guess, b_guess)

xs = np.linspace(0, 10, 2).reshape(-1, 1)
plt.scatter(X, y, s=18, alpha=0.5, label="data")
plt.plot(xs, predict(xs, w_guess, b_guess), "r-", label=f"guess: w={w_guess[0]}, b={b_guess}")
plt.xlabel("$x$"); plt.ylabel("$y$"); plt.legend(); plt.title("A bad model"); plt.show()

## 3. The cost function

We need a number that says *how bad* a given $(w, b)$ is. For each example take the
**residual** $f_{w,b}(x^{(i)}) - y^{(i)}$, square it (so over- and under-shooting both
count as error, and big mistakes hurt more), and average:

$$\boxed{\;J(w,b) = \frac{1}{2m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)^2\;}$$

This is **mean squared error**. Two details:

- **Why $\frac{1}{m}$?** So the cost doesn't grow just because you collected more data.
- **Why the extra $\frac{1}{2}$?** Pure convenience. Differentiating the square brings
  down a factor of 2 which cancels it, leaving clean gradients. It scales $J$ by a
  constant, so the location of the minimum is unchanged.

Vectorised, with $e = Xw + b - y$:

$$J = \frac{1}{2m}\, e^\top e$$

In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    error = predict(X, w, b) - y           # (m,)
    return float(error @ error / (2 * m))  # e . e  is the sum of squares


print(f"cost of our bad guess : {compute_cost(X, y, w_guess, b_guess):8.3f}")
print(f"cost of the true line : {compute_cost(X, y, np.array([3.5]), -2.0):8.3f}")

### What does $J$ look like?

Fix $b$ at its true value and vary $w$. Because the error is *squared*, $J$ is a
parabola in $w$, one minimum, no local traps. This is why linear regression with MSE
is such a friendly starting point: gradient descent cannot get stuck.

In [ ]:
ws = np.linspace(0, 7, 200)
costs = [compute_cost(X, y, np.array([w]), -2.0) for w in ws]

plt.plot(ws, costs)
plt.axvline(3.5, color="g", ls="--", label="true $w$")
plt.axvline(2.0, color="r", ls="--", label="our guess")
plt.xlabel("$w$"); plt.ylabel("$J(w, b=-2)$"); plt.legend()
plt.title("Cost is a parabola in $w$"); plt.show()

With both parameters free, $J(w,b)$ is a **bowl** (a convex quadratic surface). Contour
lines are ellipses; the centre of the ellipses is the optimum we are hunting for.

In [ ]:
w_grid = np.linspace(1.5, 5.5, 120)
b_grid = np.linspace(-8, 4, 120)
WW, BB = np.meshgrid(w_grid, b_grid)
JJ = np.array([[compute_cost(X, y, np.array([w]), b) for w in w_grid] for b in b_grid])

fig = plt.figure(figsize=(11, 4))

ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(WW, BB, JJ, cmap="viridis", alpha=0.85, linewidth=0)
ax.set_xlabel("$w$"); ax.set_ylabel("$b$"); ax.set_zlabel("$J$")
ax.set_title("The bowl")

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(WW, BB, JJ, levels=np.logspace(0.3, 3.2, 18), cmap="viridis")
ax2.clabel(cs, inline=True, fontsize=7)
ax2.plot(3.5, -2.0, "r*", markersize=14, label="minimum")
ax2.set_xlabel("$w$"); ax2.set_ylabel("$b$"); ax2.legend(); ax2.set_title("Contours")
plt.tight_layout(); plt.show()

## 4. The gradient

We want to move downhill. The **partial derivative** of $J$ with respect to a parameter
says how much $J$ changes when we nudge that parameter up. Derivation for $w_j$, using
the chain rule on the squared term:

$$
\frac{\partial J}{\partial w_j}
= \frac{\partial}{\partial w_j}\,\frac{1}{2m}\sum_{i=1}^{m}\big(f(x^{(i)}) - y^{(i)}\big)^2
= \frac{1}{2m}\sum_{i=1}^{m} 2\big(f(x^{(i)}) - y^{(i)}\big)\cdot\frac{\partial f(x^{(i)})}{\partial w_j}
$$

and since $f(x) = \sum_k w_k x_k + b$ we have $\partial f/\partial w_j = x_j$ and
$\partial f/\partial b = 1$. The 2 cancels the $\tfrac12$ (that is what it was for), giving:

$$\boxed{\;\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=1}^{m}\big(f(x^{(i)}) - y^{(i)}\big)\,x_j^{(i)}
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\big(f(x^{(i)}) - y^{(i)}\big)\;}$$

Read them out loud: *the gradient is the error, weighted by the feature that caused it.*
An example we over-predict pushes the weights down in proportion to its own feature values.

Vectorised, with $e = Xw + b - y$:

$$\nabla_w J = \frac{1}{m} X^\top e \qquad \frac{\partial J}{\partial b} = \frac{1}{m}\mathbf{1}^\top e$$

Shape check: $X^\top$ is $(n,m)$, $e$ is $(m,)$, so $X^\top e$ is $(n,)$ — the same shape
as $w$, as a gradient must be.

In [ ]:
def compute_gradient(X, y, w, b):
    m = X.shape[0]
    error = predict(X, w, b) - y      # (m,)
    dj_dw = (X.T @ error) / m         # (n, m) @ (m,) -> (n,)
    dj_db = float(error.sum() / m)
    return dj_dw, dj_db


dj_dw, dj_db = compute_gradient(X, y, w_guess, b_guess)
print(f"at w={w_guess[0]}, b={b_guess}:  dJ/dw = {dj_dw[0]:.3f}   dJ/db = {dj_db:.3f}")
print("both negative -> increasing w and b lowers the cost, which matches the plot above")

### Always check a gradient numerically

A derivative can be approximated by a **central difference**:

$$\frac{\partial J}{\partial w_j} \approx \frac{J(w_j + \epsilon) - J(w_j - \epsilon)}{2\epsilon}$$

It is far too slow to train with (one full cost evaluation per parameter per step), but
it is the single most effective way to catch a sign error or a missing $\frac{1}{m}$.
Do this every time you hand-derive a gradient.

In [ ]:
eps = 1e-6
w_up, w_dn = w_guess + eps, w_guess - eps
numeric_w = (compute_cost(X, y, w_up, b_guess) - compute_cost(X, y, w_dn, b_guess)) / (2 * eps)
numeric_b = (compute_cost(X, y, w_guess, b_guess + eps) - compute_cost(X, y, w_guess, b_guess - eps)) / (2 * eps)

print(f"dJ/dw  analytic {dj_dw[0]:12.6f}   numeric {numeric_w:12.6f}")
print(f"dJ/db  analytic {dj_db:12.6f}   numeric {numeric_b:12.6f}")

## 5. Gradient descent

Repeat until convergence:

$$
w_j := w_j - \alpha\,\frac{\partial J}{\partial w_j}
\qquad
b := b - \alpha\,\frac{\partial J}{\partial b}
$$

- The **minus** sign is the whole idea: the gradient points *uphill*, so we step against it.
- $\alpha$ is the **learning rate**: how big a step to take.
- $:=$ means assignment, and the updates must be **simultaneous**, compute *all*
  the derivatives from the current parameters first, then update. Updating $w$ and then
  using the new $w$ to compute $b$'s gradient is a classic bug:

```python
# WRONG                                  # RIGHT
w = w - alpha * dj_dw(X, y, w, b)        dj_dw, dj_db = compute_gradient(X, y, w, b)
b = b - alpha * dj_db(X, y, w, b)        w = w - alpha * dj_dw
#                    ^ already-updated w b = b - alpha * dj_db
```

This is **batch** gradient descent: every step uses all $m$ examples.

In [ ]:
def gradient_descent(X, y, w, b, alpha, num_iters):
    w = np.asarray(w, dtype=float).copy()
    b = float(b)
    history = []

    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)   # gradients at the OLD parameters
        w = w - alpha * dj_dw                          # then update both
        b = b - alpha * dj_db
        history.append((w.copy(), b, compute_cost(X, y, w, b)))

    return w, b, history


w_fit, b_fit, hist = gradient_descent(X, y, np.zeros(1), 0.0, alpha=0.01, num_iters=2000)
print(f"learned : w = {w_fit[0]:.4f}, b = {b_fit:.4f}")
print(f"true    : w = 3.5000, b = -2.0000")
print(f"cost    : {compute_cost(X, y, np.zeros(1), 0.0):.3f} -> {hist[-1][2]:.3f}")

### Watching it learn

Two views of the same run: the **learning curve** (cost per iteration — this is the plot
you look at when debugging) and the **path** the parameters take across the contours.

In [ ]:
costs_hist = [h[2] for h in hist]
path = np.array([[h[0][0], h[1]] for h in hist])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(costs_hist)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("$J(w,b)$")
axes[0].set_title("Learning curve — must go down")

axes[1].contour(WW, BB, JJ, levels=np.logspace(0.3, 3.2, 18), cmap="viridis", alpha=0.6)
axes[1].plot(path[:, 0], path[:, 1], "r.-", markersize=2, linewidth=1)
axes[1].plot(3.5, -2.0, "k*", markersize=13)
axes[1].set_xlabel("$w$"); axes[1].set_ylabel("$b$"); axes[1].set_title("Descent path")

axes[2].scatter(X, y, s=15, alpha=0.4, label="data")
for it, c in [(0, "0.8"), (5, "0.6"), (25, "0.4"), (len(hist) - 1, "r")]:
    w_i, b_i, _ = hist[it]
    axes[2].plot(xs, predict(xs, w_i, b_i), color=c, label=f"iter {it}")
axes[2].set_xlabel("$x$"); axes[2].set_ylabel("$y$"); axes[2].legend(fontsize=7)
axes[2].set_title("The line finding its way")

plt.tight_layout(); plt.show()

### Notice the elbow in the descent path

It shoots down almost vertically first, then crawls along a narrow valley. That is what
**badly scaled features** look like — $x$ ranges over $[0, 10]$ while the bias is
effectively a feature that is always $1$, so the two parameters have very different
sensitivities. Section 7 fixes exactly this.

## 6. Choosing the learning rate $\alpha$

- **too small** → correct but painfully slow;
- **too large** → the step overshoots the valley and the cost *increases*, often to `inf`/`nan`;
- **just right** → cost drops fast and smoothly, and *decreases on every single iteration*.

If your learning curve ever goes up, your first suspect is $\alpha$ (your second is a
wrong gradient).

In [ ]:
plt.figure(figsize=(6, 4))
for alpha in [0.0005, 0.005, 0.02, 0.0295]:
    _, _, h = gradient_descent(X, y, np.zeros(1), 0.0, alpha=alpha, num_iters=120)
    plt.plot([c for _, _, c in h], label=f"$\\alpha$ = {alpha}")
plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("$J$ (log scale)")
plt.legend(); plt.title("Effect of the learning rate"); plt.show()

# Past a critical value the cost grows every step until it overflows:
with np.errstate(over="ignore", invalid="ignore"):
    _, _, h_bad = gradient_descent(X, y, np.zeros(1), 0.0, alpha=0.06, num_iters=400)
bad = [c for _, _, c in h_bad]
print("alpha = 0.06, cost at iterations 0, 100, 200, 300, 399:")
print("   ", [f"{bad[i]:.3g}" for i in (0, 100, 200, 300, 399)])

## 7. Multiple features and feature scaling

Nothing in our code assumed $n = 1$ — `X @ w` already handles any number of features.
The real difficulty with many features is that they live on wildly different scales
(square feet in the thousands, number of bedrooms in single digits). The cost contours
become long thin ellipses and gradient descent zig-zags.

**Z-score normalisation** rescales each feature to mean 0 and standard deviation 1:

$$x_j^{(i)} \leftarrow \frac{x_j^{(i)} - \mu_j}{\sigma_j}
\qquad
\mu_j = \frac{1}{m}\sum_i x_j^{(i)},\quad
\sigma_j = \sqrt{\frac{1}{m}\sum_i (x_j^{(i)} - \mu_j)^2}$$

> **Important:** compute $\mu$ and $\sigma$ on the *training* set and reuse those exact
> numbers on validation/test data. Recomputing them on the test set leaks information
> about it into your pipeline.

In [ ]:
def zscore_normalize(X, mu=None, sigma=None):
    if mu is None:
        mu = X.mean(axis=0)
    if sigma is None:
        sigma = np.where(X.std(axis=0) == 0, 1.0, X.std(axis=0))
    return (X - mu) / sigma, mu, sigma


# 3 features on very different scales: size (sq ft), bedrooms, age (years)
m2 = 300
X2 = np.column_stack([
    rng.uniform(500, 4000, m2),
    rng.integers(1, 6, m2).astype(float),
    rng.uniform(0, 60, m2),
])
y2 = X2 @ np.array([0.12, 15.0, -1.4]) + 50 + rng.normal(0, 8, m2)

X2n, mu, sigma = zscore_normalize(X2)
print("raw    mean:", X2.mean(axis=0).round(1), " std:", X2.std(axis=0).round(1))
print("scaled mean:", X2n.mean(axis=0).round(6), " std:", X2n.std(axis=0).round(6))

_, _, h_raw = gradient_descent(X2, y2, np.zeros(3), 0.0, alpha=1e-7, num_iters=600)
w2, b2, h_scaled = gradient_descent(X2n, y2, np.zeros(3), 0.0, alpha=0.1, num_iters=600)

plt.plot([c for _, _, c in h_raw], label="raw features ($\\alpha=10^{-7}$, any larger diverges)")
plt.plot([c for _, _, c in h_scaled], label="scaled features ($\\alpha=0.1$)")
plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("$J$"); plt.legend()
plt.title("Feature scaling buys you a usable learning rate"); plt.show()

The scaled run converges in a few hundred iterations; the raw one is still barely moving.
Same algorithm, same data — only the coordinate system changed.

The learned weights are now in *scaled* units. To read them back in original units,
undo the scaling: $w_j^{\text{orig}} = w_j / \sigma_j$ and
$b^{\text{orig}} = b - \sum_j w_j \mu_j / \sigma_j$.

In [ ]:
w_orig = w2 / sigma
b_orig = b2 - np.sum(w2 * mu / sigma)
print("recovered weights:", w_orig.round(3), " bias:", round(b_orig, 2))
print("true weights     : [ 0.12  15.    -1.4 ]  bias: 50")

## 8. Sanity check: the closed-form solution

Linear regression is one of the very few models with an exact analytic solution. Append
a column of ones to $X$ (absorbing $b$ into the weights as $\theta_0$) and set the
gradient to zero; the result is the **normal equation**:

$$\theta = (X_b^\top X_b)^{-1} X_b^\top y$$

It needs no learning rate and no iteration — but it costs $O(n^3)$ to solve, so it is
useless once $n$ is large, and it does not generalise to models like neural networks.
Gradient descent does both. Here we use it purely as a correctness check.

(In code, always `np.linalg.solve(A, b)` rather than `np.linalg.inv(A) @ b` — same
answer, better numerics, faster.)

In [ ]:
def normal_equation(X, y):
    X_b = np.hstack([np.ones((X.shape[0], 1)), X])
    theta = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
    return theta[1:], float(theta[0])


w_exact, b_exact = normal_equation(X, y)
print(f"gradient descent : w = {w_fit[0]:.5f}   b = {b_fit:.5f}")
print(f"normal equation  : w = {w_exact[0]:.5f}   b = {b_exact:.5f}")

## 9. The same thing, from the module

`linear_regression.py` packages all of this behind a small class, so later lessons can
just import it.

In [ ]:
from linear_regression import LinearRegression

model = LinearRegression(alpha=0.1, num_iters=2000, normalize=True).fit(X2, y2)
print("R^2 on training data:", round(model.score(X2, y2), 4))
print("prediction for a 2500 sq ft, 3 bed, 10 year old house:",
      round(model.predict(np.array([[2500.0, 3.0, 10.0]]))[0], 1))

## Exercises

1. **Break it on purpose.** Remove the `.copy()` in `gradient_descent`, or update `b`
   using the already-updated `w`. What changes, and does the numerical gradient check
   still pass?
2. **Mean absolute error.** Swap the squared error for $\frac{1}{m}\sum|f(x^{(i)}) - y^{(i)}|$.
   Derive its gradient (careful at zero), implement it, and compare how the two costs
   react when you add one wildly wrong outlier to the data.
3. **Convergence test.** Stop early when $J$ improves by less than $10^{-9}$ between
   iterations instead of running a fixed `num_iters`. How many iterations does each
   $\alpha$ from section 6 actually need?
4. **Polynomial features.** Fit $y = 0.5x^2 + 3$ by feeding the model a second column
   $x^2$. The model is still *linear in its parameters* — that is the only linearity
   linear regression requires.
5. **Stochastic gradient descent.** Update using one random example at a time instead of
   all $m$. Plot the learning curve — why is it noisy, and does it still get there?

## What's next

Lesson 02 — **logistic regression**: the same machinery (cost → gradient → descent) with
a sigmoid squashing the output into $(0,1)$ and cross-entropy replacing squared error.